In [12]:
import pandas as pd
import datasets
from pathlib import Path

In [2]:
# pull dataset from huggingface
train_dataset = datasets.load_dataset("VijayRam1812/safety_dataset", split="train")
val_dataset = datasets.load_dataset("VijayRam1812/safety_dataset", split="val")
test_dataset = datasets.load_dataset("VijayRam1812/safety_dataset_synthetic_eval", split="train")


In [3]:
# convert to pandas dataframe
train_df = train_dataset.to_pandas()
val_df = val_dataset.to_pandas()
test_df = test_dataset.to_pandas()

In [4]:
train_df.head()

,messages,label,source,conv_type,raw_text,category,stratify_key
0,[{'content': 'My friends and I are huge music ...,1,pku,single_turn,"USER: My friends and I are huge music fans, bu...",IP,13
1,[{'content': 'What's the earliest age it's app...,0,anthropic_hh,multi_turn,USER: What's the earliest age it's appropriate...,CSAE,1
2,"[{'content': 'The show, the Sopranos, famously...",1,anthropic_hh,multi_turn,"USER: The show, the Sopranos, famously had a p...",CSAE,10
3,[{'content': 'Can you help me download the lat...,1,generated,user_only,USER: Can you help me download the latest epis...,IP,13
4,[{'content': 'Can you tell me how to overdose ...,1,generated,single_turn,USER: Can you tell me how to overdose on commo...,SHS,16


In [5]:
val_df.head()

,messages,label,source,conv_type,raw_text,category,stratify_key
0,[{'content': 'I want to steal money from a KFC...,0,anthropic_hh,multi_turn,USER: I want to steal money from a KFC cash re...,SCAM,6
1,[{'content': 'My daughter is reading the book ...,1,anthropic_hh,multi_turn,"USER: My daughter is reading the book ""Lord of...",CSAE,10
2,"[{'content': 'You make me so angry', 'role': '...",0,pku,single_turn,USER: You make me so angry\nASSISTANT: That's ...,SHS,7
3,[{'content': 'Can you describe a method to exp...,0,generated,single_turn,USER: Can you describe a method to exploit chi...,CSAE,1
4,[{'content': 'is it okay to take money from my...,0,anthropic_hh,multi_turn,USER: is it okay to take money from my partner...,SCAM,6


In [6]:
# Sample equal counts per (stratify_key, conv_type) cell for balanced conv_type distribution
n_cells = train_df.groupby(["stratify_key", "conv_type"]).ngroups
n_per_cell = max(1, len(train_df) // 2 // n_cells)
min_available = train_df.groupby(["stratify_key", "conv_type"]).size().min()
n_per_cell = min(n_per_cell, min_available)

train_sampled = train_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(
    lambda g: g.sample(n=n_per_cell, random_state=42)
).reset_index(drop=True)

print(f"Original training set: {len(train_df)} samples")
print(f"Sampled training set:  {len(train_sampled)} samples  ({n_per_cell} per cell)")
print(f"\nPer stratify_key counts (before -> after):")
comparison = pd.DataFrame({
    "original": train_df["stratify_key"].value_counts().sort_index(),
    "sampled":  train_sampled["stratify_key"].value_counts().sort_index()
})
comparison["ratio"] = (comparison["sampled"] / comparison["original"]).round(2)
print(comparison.to_string())


Original training set: 28382 samples
Sampled training set:  14148 samples  (262 per cell)

Per stratify_key counts (before -> after):
              original  sampled  ratio
stratify_key                          
0                 1577      786    0.5
1                 1577      786    0.5
2                 1577      786    0.5
3                 1577      786    0.5
4                 1577      786    0.5
5                 1576      786    0.5
6                 1577      786    0.5
7                 1577      786    0.5
8                 1577      786    0.5
9                 1577      786    0.5
10                1577      786    0.5
11                1577      786    0.5
12                1577      786    0.5
13                1577      786    0.5
14                1576      786    0.5
15                1577      786    0.5
16                1576      786    0.5
17                1576      786    0.5


C:\Users\vjayr\AppData\Local\Temp\ipykernel_21668\3531789979.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_sampled = train_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(


In [7]:
# find the distribution of conv_types in the sampled data
conv_type_counts = train_sampled["conv_type"].value_counts()
conv_type_percentages = (conv_type_counts / len(train_sampled) * 100).round(2)
print("\nDistribution of conv_types in the sampled training data:")
for conv_type, count in conv_type_counts.items():
    pct = conv_type_percentages[conv_type]
    print(f"{conv_type}: {count} samples ({pct}%)")


Distribution of conv_types in the sampled training data:
multi_turn: 4716 samples (33.33%)
single_turn: 4716 samples (33.33%)
user_only: 4716 samples (33.33%)


In [8]:
# Sample equal counts per (stratify_key, conv_type) cell for balanced conv_type distribution
n_cells = val_df.groupby(["stratify_key", "conv_type"]).ngroups
n_per_cell = max(1, len(val_df) // 2 // n_cells)
min_available = val_df.groupby(["stratify_key", "conv_type"]).size().min()
n_per_cell = min(n_per_cell, min_available)

val_sampled = val_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(
    lambda g: g.sample(n=n_per_cell, random_state=42)
).reset_index(drop=True)

print(f"Original validation set: {len(val_df)} samples")
print(f"Sampled validation set:  {len(val_sampled)} samples  ({n_per_cell} per cell)")
print(f"\nPer stratify_key counts (before -> after):")
comparison = pd.DataFrame({
    "original": val_df["stratify_key"].value_counts().sort_index(),
    "sampled":  val_sampled["stratify_key"].value_counts().sort_index()
})
comparison["ratio"] = (comparison["sampled"] / comparison["original"]).round(2)
print(comparison.to_string())

Original validation set: 3154 samples
Sampled validation set:  1566 samples  (29 per cell)

Per stratify_key counts (before -> after):
              original  sampled  ratio
stratify_key                          
0                  175       87   0.50
1                  175       87   0.50
2                  175       87   0.50
3                  175       87   0.50
4                  175       87   0.50
5                  176       87   0.49
6                  175       87   0.50
7                  175       87   0.50
8                  175       87   0.50
9                  175       87   0.50
10                 175       87   0.50
11                 175       87   0.50
12                 175       87   0.50
13                 175       87   0.50
14                 176       87   0.49
15                 175       87   0.50
16                 176       87   0.49
17                 176       87   0.49


C:\Users\vjayr\AppData\Local\Temp\ipykernel_21668\589419558.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sampled = val_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(


In [9]:
# find the distribution of conv_types in the sampled data
conv_type_counts = val_sampled["conv_type"].value_counts()
conv_type_percentages = (conv_type_counts / len(val_sampled) * 100).round(2)
print("\nDistribution of conv_types in the sampled training data:")
for conv_type, count in conv_type_counts.items():
    pct = conv_type_percentages[conv_type]
    print(f"{conv_type}: {count} samples ({pct}%)")


Distribution of conv_types in the sampled training data:
multi_turn: 522 samples (33.33%)
single_turn: 522 samples (33.33%)
user_only: 522 samples (33.33%)


In [13]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train_dataset.csv", index=False)
val_df.to_csv(output_dir / "val_dataset.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  train_dataset.csv  : {len(train_df)} rows")
print(f"  val_dataset.csv    : {len(val_df)} rows")

Saved to C:\Vijay\PyCode\ContentID\data\raw
  train_dataset.csv  : 28382 rows
  val_dataset.csv    : 3154 rows


In [ ]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "train"
output_dir.mkdir(parents=True, exist_ok=True)

train_sampled.to_csv(output_dir / "train_sampled.csv", index=False)
val_sampled.to_csv(output_dir / "val_sampled.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  train_sampled.csv  : {len(train_sampled)} rows")
print(f"  val_sampled.csv    : {len(val_sampled)} rows")


Saved to C:\Vijay\PyCode\ContentID\data\train
  train_sampled.csv  : 14148 rows
  val_sampled.csv    : 1566 rows


In [11]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "test"
output_dir.mkdir(parents=True, exist_ok=True)

test_df.to_csv(output_dir / "test_dataset.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  test_dataset.csv  : {len(test_df)} rows")

Saved to C:\Vijay\PyCode\ContentID\data\test
  test_dataset.csv  : 3780 rows
